# HydraY NNUE: threat inputs recipes on v10 (selectable `ARM`)

Derived from `colab_v10.ipynb`, `colab_recipe.ipynb` and `colab_TEST_probe_threats.ipynb`
for `threat-inputs`. Runtime -> **G4 (NVIDIA RTX PRO 6000)** (~75 GiB free disk), set `ARM`
in the cell "configuration", then "Run all".

- **Branch**: `threat-inputs`
- **Architecture**: `768x4hm + threats (70,050) -> 1024 -> 16 -> 1`, `8` output buckets (`78,156,864 B` quantised file)
- **Eval scale**: `356` (`400 / 1.124`: v9's targets in v10's label unit)
- **Pool**: `v10` (`1,954,186,654` records in 16 parts, sha256-checked; ~10.2 epochs per 200 SB)

### Arms
`ARM` selects one recipe in the configuration cell (`STAGE=20` throughout):

| `ARM` | phases | total SB | schedule |
|---|---|---:|---|
| `long-400` *(default)* | `antidead-long` (`320`) -> `ft-antidead-long-qat` (`80`) | `400` (~20.5 ep) | Fase 1 `steps`: `1e-3` (sb 1-60) -> `1e-4` (sb 61-240) -> `3e-5` (sb 241-320), `l1b_init=0.1`, `warmup=3000`, `float_clip=64`; Fase 2 `cosine` `5e-5 -> 2.5e-6`, `warmup=3000`, `revive`, `QAT` |
| `160-80` | `antidead-steps` (`160`) -> `ft-antidead-steps-qat` (`80`) | `240` (~12.3 ep) | Fase 1 `steps` (scalata 1:2 da `long-400`): `1e-3` (sb 1-30) -> `1e-4` (sb 31-120) -> `3e-5` (sb 121-160), `l1b_init=0.1`, `warmup=3000`, `float_clip=64`; Fase 2 `cosine` `5e-5 -> 2.5e-6`, `warmup=3000`, `revive`, `QAT` |
| `antidead-qat` | `antidead` (`160`) -> `ft-antidead-qat` (`40`) | `200` (~10.2 ep) | Ricetta originale: Fase 1 StepLR `1e-3`, `x0.1` a sb 80; Fase 2 `cosine` `1e-4 -> 2.5e-6`, `revive`, `QAT` |
| `ft-from-160` | `ft-from160-qat` (`80`) | `80` (~4.1 ep) | Sola Fase 2 lunga (`cosine` `5e-5 -> 2.5e-6`, `revive`, `QAT`) da `hydray-threats-v10-antidead-160sb.nnue` su Drive |

**Disk & Drive notes**: each threat checkpoint directory is ~1.2 GiB (vs ~100 MiB without threats).
To stay inside G4's ~75 GiB local disk alongside the ~59 GiB v10 pool, local superseded checkpoints
are pruned as soon as the new stage checkpoint is copied and verified on Drive. Keeping all stage
checkpoints (`STAGE=20`) uses ~12 GiB on Drive for `antidead-qat` (10 ckpts) or ~24 GiB for `long-400`
(20 ckpts); set `PRUNE_OLD_DRIVE_CKPTS = True` if Drive space is tight.

In [ ]:
# --- helper: any failed command stops the notebook, output streams live ---
import subprocess, os, re, glob, hashlib, shutil, shlex

def sh(cmd, check=True):
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash',
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    out = []
    for line in p.stdout:
        print(line, end='', flush=True)
        out.append(line)
    if p.wait() != 0 and check:
        raise RuntimeError(f'FAILED (exit {p.returncode}): {cmd}')
    return ''.join(out)

gpu = subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader',
                     shell=True, capture_output=True, text=True).stdout.strip()
print('GPU:', gpu or 'none')
sh('nproc; free -g | head -2; df -h /content | tail -1')

In [ ]:
# --- configuration ---
import json, time, urllib.request
from concurrent.futures import ThreadPoolExecutor

if not os.path.ismount('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive'

ARM    = 'long-400'      # long-400 | 160-80 | antidead-qat | ft-from-160
STAGE  = 20              # checkpoint to Drive every STAGE superbatches (multiple of 10)
BRANCH = 'threat-inputs'
BUCKETS = 8              # the 4.2.0 layout
EVAL_SCALE = '356'       # 400 / 1.124: v9's targets at the recipe's 400 (label unit)
DOWNLOADS  = 8           # parts downloaded at once
PRUNE_OLD_DRIVE_CKPTS = False  # True keeps only the latest Drive checkpoint of the current phase (~1.4 GiB instead of ~12-24 GiB)

REPO    = '/content/hydray'
TRAINER = f'{REPO}/nnue/trainer'
LOCAL   = '/content/data'
INIT_NET = f'{REPO}/nnue/net/hydray.nnue'          # sha256-checked reference net (4.4.0)
INIT_SHA = '844d3ca82c3c4cf41e354bee866726e45ef5ac8ab2cd2c1786fe0a6cc8f4fa83'
HF_V10   = 'https://huggingface.co/datasets/ThomasGhione/hydray/resolve/main/v10'
V10_RECORDS = 1_954_186_654

def net_id(phase, sb):
    return f'hydray-threats-v10-{phase}-{sb}sb'

ANTIDEAD = {'HYDRAY_L1B_INIT': '0.1', 'HYDRAY_WARMUP': '3000', 'HYDRAY_FLOAT_CLIP': '64'}
ANTIDEAD_LONG = dict(ANTIDEAD, HYDRAY_LR='steps', HYDRAY_LR_STEPS='60:0.0001,240:0.00003')
ANTIDEAD_160_STEPS = dict(ANTIDEAD, HYDRAY_LR='steps', HYDRAY_LR_STEPS='30:0.0001,120:0.00003')
FT = {'HYDRAY_LR': 'cosine', 'HYDRAY_LR_START': '0.0001', 'HYDRAY_LR_FINAL': '0.0000025',
      'HYDRAY_WARMUP': '3000', 'HYDRAY_FLOAT_CLIP': '64', 'HYDRAY_REVIVE': '1', 'HYDRAY_L1B_INIT': '0.1'}
FT_LONG = dict(FT, HYDRAY_LR_START='0.00005')

ARMS = {
    'antidead-qat': [
        ('antidead',        160, ANTIDEAD),
        ('ft-antidead-qat', 40,  dict(FT, HYDRAY_QAT='1', HYDRAY_INIT_NET=f"{DRIVE}/{net_id('antidead', 160)}.nnue")),
    ],
    '160-80': [
        ('antidead-steps',        160, ANTIDEAD_160_STEPS),
        ('ft-antidead-steps-qat', 80,  dict(FT_LONG, HYDRAY_QAT='1', HYDRAY_INIT_NET=f"{DRIVE}/{net_id('antidead-steps', 160)}.nnue")),
    ],
    'long-400': [
        ('antidead-long',        320, ANTIDEAD_LONG),
        ('ft-antidead-long-qat', 80,  dict(FT_LONG, HYDRAY_QAT='1', HYDRAY_INIT_NET=f"{DRIVE}/{net_id('antidead-long', 320)}.nnue")),
    ],
    'ft-from-160': [
        ('ft-from160-qat', 80, dict(FT_LONG, HYDRAY_QAT='1', HYDRAY_INIT_NET=f"{DRIVE}/{net_id('antidead', 160)}.nnue")),
    ],
}
assert ARM in ARMS, f'unknown ARM {ARM}'
PHASES = ARMS[ARM]
if EVAL_SCALE != '400':
    PHASES = [(p, sb, dict(env, HYDRAY_EVAL_SCALE=EVAL_SCALE)) for p, sb, env in PHASES]

def use_phase(i):
    """Makes phase i current: the stage helpers read these globals."""
    global PHASE, TOTAL_SB, ENV, NET_ID, ENV_PREFIX
    PHASE, TOTAL_SB, ENV = PHASES[i]
    NET_ID = net_id(PHASE, TOTAL_SB)
    ENV_PREFIX = ' '.join(f'{k}={shlex.quote(str(v))}' for k, v in ENV.items())

assert gpu, 'no GPU: change the runtime type to G4 (RTX PRO 6000)'
if 'RTX PRO 6000' not in gpu:
    print(f'WARNING: {gpu} is not the RTX PRO 6000 this notebook is set up for')
assert STAGE % 10 == 0 and all(sb % STAGE == 0 for _, sb, _ in PHASES), \
    'stage ends must be checkpoints (save_rate 10)'
if 'HYDRAY_INIT_NET' in PHASES[0][2]:
    init0 = PHASES[0][2]['HYDRAY_INIT_NET']
    assert os.path.exists(init0), f'ARM={ARM} needs {init0} on Drive'

NET_FILE_BYTES = 78_156_864
TRAINER_LINE   = f'trainer_deep: 768x4hm + threats -> 1024 -> 16 -> 1, {BUCKETS} output buckets'

MANIFEST = json.loads(urllib.request.urlopen(f'{HF_V10}/manifest.json').read())
assert MANIFEST['records'] == V10_RECORDS and not MANIFEST['test_bytes'], 'not the v10 manifest'
print(f"arm {ARM}, branch {BRANCH}, v10: {MANIFEST['records']:,} records in {len(MANIFEST['parts'])} parts, "
      f"holdout {MANIFEST['holdout']['records']:,}")
for i in range(len(PHASES)):
    use_phase(i)
    print(f'  phase {PHASE}: {TOTAL_SB} superbatches, net id {NET_ID}, environment {ENV}')

In [ ]:
# CUDA toolkit for this GPU: NVRTC compiles bullet's kernels for sm_<compute capability>,
# and a Blackwell GPU (12.x) needs CUDA >= 12.8. An older toolkit gets the 12.8 libraries
# next to it (NVIDIA's apt repository), and the build and the trainer use those.
cc = subprocess.run('nvidia-smi --query-gpu=compute_cap --format=csv,noheader', shell=True,
                    capture_output=True, text=True).stdout.split()[0]
def toolkit(path):
    try:
        v = json.load(open(f'{path}/version.json'))['cuda']['version']
    except (OSError, KeyError, ValueError):
        out = subprocess.run(f'{path}/bin/nvcc --version', shell=True, capture_output=True, text=True).stdout
        m = re.search(r'release (\d+)\.(\d+)', out)
        v = f'{m.group(1)}.{m.group(2)}' if m else '0.0'
    return tuple(int(x) for x in v.split('.')[:2])
NEED = (12, 8) if int(cc.split('.')[0]) >= 12 else (11, 0)
CUDA = '/usr/local/cuda'
print(f'compute capability {cc}, toolkit {toolkit(CUDA)} at {CUDA}, needed {NEED}')
if toolkit(CUDA) < NEED:
    distro = subprocess.run('. /etc/os-release && echo ubuntu${VERSION_ID/./}', shell=True,
                            capture_output=True, text=True, executable='/bin/bash').stdout.strip()
    sh(f'cd /tmp && wget -q https://developer.download.nvidia.com/compute/cuda/repos/{distro}/x86_64/cuda-keyring_1.1-1_all.deb'
       ' && dpkg -i cuda-keyring_1.1-1_all.deb >/dev/null && apt-get -qq update'
       ' && apt-get -qq install -y cuda-nvrtc-dev-12-8 cuda-cudart-dev-12-8 libcublas-dev-12-8 cuda-driver-dev-12-8 >/dev/null')
    CUDA = '/usr/local/cuda-12.8'
    # bullet's build.rs links from CUDA_PATH/lib64.
    if not os.path.exists(f'{CUDA}/lib64'):
        os.symlink(f'{CUDA}/targets/x86_64-linux/lib', f'{CUDA}/lib64')
    assert all(os.path.exists(f'{CUDA}/lib64/lib{l}.so') for l in ('nvrtc', 'cudart', 'cublas')), \
        f'the CUDA 12.8 libraries are not in {CUDA}/lib64'
print('CUDA_PATH', CUDA)

# --- Rust, clone, checks, build ---
sh("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
sh(f'rm -rf {REPO} && git clone --depth 1 --branch {BRANCH} https://github.com/ThomasGhione/HydraY {REPO}')
sh(f'cd {REPO} && git log --oneline -1')

# Not ceremonial: verifies architecture, recipe knobs, QAT and threat inputs on the cloned branch.
tr = open(f'{TRAINER}/src/bin/trainer_deep.rs').read()
sd = open(f'{TRAINER}/src/bin/sanity_deep.rs').read()
assert 'const HIDDEN_SIZE: usize = 1024;' in tr and 'const L1_SIZE: usize = 16;' in tr
assert 'HYDRAY_INIT_NET' in tr and 'HYDRAY_FLOAT_CLIP' in tr, 'trainer_deep.rs has no recipe knobs: wrong branch'
assert 'HYDRAY_QAT' in tr and 'PAIRWISE_NUDGE' in sd, 'trainer_deep.rs has no QAT: wrong branch'
if any(env.get('HYDRAY_LR') == 'steps' for _, _, env in PHASES):
    assert 'HYDRAY_LR_STEPS' in tr, 'trainer_deep.rs has no HYDRAY_LR_STEPS: push local trainer_deep.rs to threat-inputs first'
assert 'pub fn to_bullet' in sd and 'pub fn dead_rows' in sd
assert 'threats::FEATURES' in tr, f'branch {BRANCH} has no threat inputs'
assert hashlib.sha256(open(INIT_NET, 'rb').read()).hexdigest() == INIT_SHA, 'nnue/net/hydray.nnue is not the shipped net (4.4.0)'

CARGO = f'cd {TRAINER} && PATH=$HOME/.cargo/bin:$PATH env CUDA_PATH={CUDA} LD_LIBRARY_PATH={CUDA}/lib64:$LD_LIBRARY_PATH HYDRAY_OUTPUT_BUCKETS={BUCKETS}'
sh(f'{CARGO} cargo build -r --bin trainer_deep --features cuda')
sh(f'{CARGO} cargo build -r --bin sanity_deep')
sh(f'{TRAINER}/target/release/sanity_deep {INIT_NET} --init-check')

# The activation census (C++, piece-only). For threat inputs l1census does not load l0t,
# so only sanity_deep's zero-row count runs on threat checkpoints.
L1CENSUS = '/content/l1census'
print('l1census: skipped for threat inputs (zero rows via sanity_deep only)')

In [ ]:
# --- data: the 16 parts of v10 and its holdout, in parallel, each checked by sha256 ---
# A part already on local disk with the right size and checked sha256 (a restart in the
# same session) is not downloaded again. A broken download resumes where it stopped.
os.makedirs(LOCAL, exist_ok=True)
WANT = [(p['file'], p['bytes'], p['sha256']) for p in MANIFEST['parts']]
HOLD = MANIFEST['holdout']
WANT.append((HOLD['file'], 32 * HOLD['records'], HOLD['sha256']))

need = sum(size for _, size, _ in WANT)
have = sum(os.path.getsize(f'{LOCAL}/{n}') for n, _, _ in WANT if os.path.exists(f'{LOCAL}/{n}'))
st = os.statvfs('/content'); free = st.f_bavail * st.f_frsize
print(f'free {free/2**30:.1f} GiB, still to download {(need - have)/2**30:.1f} GiB')
assert free > (need - have) + 8 * 2**30, 'not enough disk: use a G4 runtime (or one with more disk)'

def fetch(item):
    name, size, sha = item
    path, ok = f'{LOCAL}/{name}', f'{LOCAL}/{name}.sha256ok'
    if os.path.exists(path) and os.path.getsize(path) == size \
            and os.path.exists(ok) and open(ok).read() == sha:
        return f'{name}: already here'
    t0 = time.time()
    for attempt in range(10):
        if os.path.exists(path) and os.path.getsize(path) > size:
            os.remove(path)
        if not (os.path.exists(path) and os.path.getsize(path) == size):
            subprocess.run(['curl', '-sfL', '--retry', '5', '-C', '-', '-o', path, f'{HF_V10}/{name}'])
        if os.path.exists(path) and os.path.getsize(path) == size:
            break
        time.sleep(10 * (attempt + 1))
    got = os.path.getsize(path) if os.path.exists(path) else 0
    assert got == size, f'{name}: {got} bytes after 10 attempts, expected {size}'
    digest = subprocess.run(['sha256sum', path], capture_output=True, text=True, check=True).stdout.split()[0]
    if digest != sha:
        os.remove(path)
        raise AssertionError(f'{name}: sha256 {digest}, expected {sha} (removed: run the cell again)')
    open(ok, 'w').write(sha)
    return f'{name}: {size/2**30:.2f} GiB in {(time.time() - t0)/60:.1f} min, sha256 checked'

t0 = time.time()
with ThreadPoolExecutor(DOWNLOADS) as ex:
    for line in ex.map(fetch, WANT):
        print(line, flush=True)
print(f'all {len(WANT)} files in {(time.time() - t0)/60:.1f} min')

DATA_FILES = [f'{LOCAL}/{p["file"]}' for p in MANIFEST['parts']]
DATA = ','.join(DATA_FILES)
SAMPLE = f'{LOCAL}/{HOLD["file"]}'
total = sum(os.path.getsize(f) for f in DATA_FILES) // 32
assert total == V10_RECORDS
sbs = sum(sb for _, sb, _ in PHASES)
print(f'\ntraining stream: {total/1e9:.3f}B records, {sbs} superbatches = {sbs*6104*16384/total:.2f} epochs')

In [ ]:
# --- stage helpers and census (RUN THIS ALSO WHEN RESTARTING) ---

def census(qbin):
    """Dead l1 neurons per output bucket: all-zero rows."""
    sh(f'{TRAINER}/target/release/sanity_deep {qbin} | grep "dead l1"')
    if os.path.exists(L1CENSUS):
        sh(f'{L1CENSUS} {qbin} {SAMPLE} 0 600000')

def stage_cmd(end, start, resume_from):
    # STAGE_END and the recipe variables must sit right in front of cargo (after `env`):
    # `STAGE_END=40 cd dir && cargo ...` gives the variable to `cd` only.
    args = f'{DATA} {TOTAL_SB} {NET_ID}'
    if resume_from is not None:
        args += f' {start} checkpoints/{NET_ID}-{resume_from}'
    return f'{CARGO} {ENV_PREFIX} STAGE_END={end} cargo run -r --bin trainer_deep --features cuda -- {args}'

def check_output(out, first_stage):
    assert TRAINER_LINE in out, f'the trainer did not print "{TRAINER_LINE}": wrong build'
    recipe = re.search(r'^recipe: .*$', out, re.M)
    assert recipe, 'no recipe line: wrong branch'
    for v in ENV.values():
        assert v in recipe.group(0), f'recipe line misses {v}: {recipe.group(0)}'
    assert ('revive true' in recipe.group(0)) == ('HYDRAY_REVIVE' in ENV)
    assert ('qat true' in recipe.group(0)) == ('HYDRAY_QAT' in ENV)
    if first_stage and 'HYDRAY_INIT_NET' in ENV:
        assert out.count('init check ') == 5, 'the start check did not run'

def check_checkpoint(end):
    q = f'{TRAINER}/checkpoints/{NET_ID}-{end}/quantised.bin'
    assert os.path.exists(q), f'missing {q}'
    assert os.path.getsize(q) == NET_FILE_BYTES, f'{q}: {os.path.getsize(q)} bytes, expected {NET_FILE_BYTES}'
    return q

def save_to_drive(end):
    # Copies the checkpoint to Drive and CHECKS it arrived: the Drive mount writes
    # through a cache, so an unfinished upload would pass for a finished one.
    ck  = f'{TRAINER}/checkpoints/{NET_ID}-{end}'
    dst = f'{DRIVE}/{NET_ID}-{end}'
    sh(f'rm -rf "{dst}" && cp -r {ck} {DRIVE}/')
    size = lambda p: sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(p) for f in fs)
    assert os.path.isdir(dst) and size(dst) == size(ck), f'incomplete copy on Drive: {dst}'
    print(f'superbatch {end} on Drive ({size(dst)/2**20:.0f} MiB, checked)', flush=True)

def prune_local_checkpoints(keep=None):
    # Threat checkpoints are ~1.2 GiB each: drop local ones once backed up on Drive
    # so 20 superbatch saves (~24 GiB total) never fill the ~75 GiB G4 disk.
    for d in sorted(glob.glob(f'{TRAINER}/checkpoints/{NET_ID}-*')):
        if keep is None or os.path.basename(d) != f'{NET_ID}-{keep}':
            shutil.rmtree(d)

def run_stages(done=0):
    assert done % STAGE == 0, f'{done} is not a stage boundary'
    prev = done if done else None
    for k in range(done // STAGE, TOTAL_SB // STAGE):
        end, start = (k + 1) * STAGE, k * STAGE + 1
        print(f'\n===== {PHASE}, stage {k+1}/{TOTAL_SB//STAGE}: superbatches {start}-{end} =====', flush=True)
        out = sh(stage_cmd(end, start, prev))
        check_output(out, prev is None)
        census(check_checkpoint(end))
        save_to_drive(end)
        prune_local_checkpoints(keep=end)
        if PRUNE_OLD_DRIVE_CKPTS and prev is not None:
            shutil.rmtree(f'{DRIVE}/{NET_ID}-{prev}', ignore_errors=True)
        prev = end

def finish():
    """Final checks of the current phase, and its net on Drive (the next phase starts from it)."""
    final = check_checkpoint(TOTAL_SB)
    sh(f'{TRAINER}/target/release/sanity_deep {final}')
    census(final)
    sh(f'cp {final} "{DRIVE}/{NET_ID}.nnue" && cmp {final} "{DRIVE}/{NET_ID}.nnue"')
    prune_local_checkpoints(keep=None)
    print(f'\nnet on Drive: {DRIVE}/{NET_ID}.nnue')
    print('Sanity: startpos small, the mirrored pair (wN e4 / bN e5) identical, a queen up ~ +1000 or more.')

def run_arm(first=0, done=0):
    """Runs the phases from `first` on; `done` superbatches of phase `first` are already trained."""
    for i in range(first, len(PHASES)):
        use_phase(i)
        if done == 0 and os.path.exists(f'{DRIVE}/{NET_ID}.nnue'):
            print(f'{NET_ID}.nnue already on Drive: phase {PHASE} skipped', flush=True)
            continue
        run_stages(done=done)
        finish()
        done = 0

In [ ]:
# --- training ---
# Do NOT run this cell when restarting: use the cell "RESTART" below.
census(INIT_NET)                                  # 4.4.0 reference net for comparison
run_arm()

In [ ]:
# --- phase nets on Drive ---
# Each phase is checked and copied to Drive by run_arm (finish), the last one included.
for i in range(len(PHASES)):
    use_phase(i)
    net = f'{DRIVE}/{NET_ID}.nnue'
    print(f'{net}: {"on Drive" if os.path.exists(net) else "MISSING"}')

In [ ]:
# --- RESTART (ONLY if the session died mid-run) ---
#   1. run the cells from "helper" to "stage helpers" (the data cell reuses the parts
#      already on local disk, or downloads them again in a new session);
#   2. do NOT run the training cell;
#   3. set RESTART = True, PHASE_ID = the phase that was running and DONE = its last
#      superbatch saved on Drive (or 0 if phase 1 finished and phase 2 had not reached stage 1).
# With RESTART = False this cell does nothing, so "Run all" is safe.
RESTART  = False
PHASE_ID = PHASES[0][0]
DONE     = 20

if not RESTART:
    print('restart off (RESTART = False): nothing to do')
else:
    first = [p[0] for p in PHASES].index(PHASE_ID)
    use_phase(first)
    if DONE > 0:
        ck = f'{DRIVE}/{NET_ID}-{DONE}'
        assert os.path.isdir(ck), f'checkpoint not found on Drive: {ck}'
        os.makedirs(f'{TRAINER}/checkpoints', exist_ok=True)
        sh(f'rm -rf "{TRAINER}/checkpoints/{NET_ID}-{DONE}" && cp -r "{ck}" {TRAINER}/checkpoints/')
        check_checkpoint(DONE)
    run_arm(first, DONE)

## After the run, on the laptop

The final net on Drive (`78,156,864 B`) depends on `ARM`:

- `long-400`: `hydray-threats-v10-ft-antidead-long-qat-80sb.nnue` (from `hydray-threats-v10-antidead-long-320sb.nnue`)
- `160-80`: `hydray-threats-v10-ft-antidead-steps-qat-80sb.nnue` (from `hydray-threats-v10-antidead-steps-160sb.nnue`)
- `antidead-qat`: `hydray-threats-v10-ft-antidead-qat-40sb.nnue` (from `hydray-threats-v10-antidead-160sb.nnue`)
- `ft-from-160`: `hydray-threats-v10-ft-from160-qat-80sb.nnue` (from `hydray-threats-v10-antidead-160sb.nnue`)

`nnue/tools/reorder.cpp` supports threat nets (8 or 16 output buckets, `.fen` or `.bin` pool, `max_positions` optional — default 20,000 for `.bin`), whereas `l1census`, `holdoutloss`, and `qatgap` still refuse threat nets on `threat-inputs`. Test the reordered net (`<NET>-reordered.nnue` below) against `4.4.0` (`nnue/net/hydray.nnue` on the `dev` binary, so `dev` does not pay the threat overhead):

**1. Reorder L0 neurons and sanity check** (on `threat-inputs`):

    g++ -std=c++23 -O2 -march=native nnue/tools/reorder.cpp nnue/nnue_deep.cpp -o /tmp/reorder
    /tmp/reorder <NET>.nnue ../v10-data/hydray_v10.holdout.bin <NET>-reordered.nnue   # or a .fen file
    ./chess nnue-selftest <NET>-reordered.nnue 200

**2. SPRT at fixed nodes** (pure eval quality vs `4.4.0`, e.g. `nodes=150000`):

    ./chess_threats_opt ... EvalFile=$PWD/<NET>-reordered.nnue vs ./chess_dev (4.4.0)

**3. SPRT at time controls (`4+0.04` and `10+0.1`, `Threads=1`)** (net Elo gain after engine cost vs `4.4.0`):

    fastchess with chess_threats_opt (EvalFile=$PWD/<NET>-reordered.nnue) vs chess_dev (4.4.0), CPUs 4-11, concurrency 8

Do not SPRT a checkpoint taken before the end of its LR schedule.